# Preprocesamiento y PCA — Titanic
Limpieza, codificación, escalado, análisis de componentes principales y exportación del dataset procesado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

sns.set_theme(style='whitegrid')

carpeta_imagenes = 'images'
carpeta_datos = 'data'
os.makedirs(carpeta_imagenes, exist_ok=True)

ruta_entrada = 'data/titanic.csv'
ruta_salida = 'data/titanic_preprocesado.csv'

datos = pd.read_csv(ruta_entrada)
print('Tamaño original:', datos.shape)


## 1. Imputación
- **Age:** mediana (robusta al sesgo). Se usa la mediana global por simplicidad y reproducibilidad; alternativa válida: mediana por grupo Pclass+Sex.
- **Embarked:** moda (solo 2 faltantes, valor dominante S).
- **Fare:** mediana preventiva (en este archivo no hay faltantes, pero se deja el paso para robustez).
- **Cabin:** 77 % faltante → no imputable; se crea el indicador binario **HasCabin** y se elimina la columna original.

In [ ]:
print('Faltantes antes:')
print(datos.isnull().sum())

mediana_edad = datos['Age'].median()
datos['Age'] = datos['Age'].fillna(mediana_edad)

moda_embarque = datos['Embarked'].mode()[0]
datos['Embarked'] = datos['Embarked'].fillna(moda_embarque)

mediana_tarifa = datos['Fare'].median()
datos['Fare'] = datos['Fare'].fillna(mediana_tarifa)

datos['HasCabin'] = datos['Cabin'].notnull().astype(int)

print('\nFaltantes despues:')
print(datos.isnull().sum())

print('\nDistribucion de HasCabin:')
print(datos['HasCabin'].value_counts())

## 2. Eliminación de columnas
Se eliminan **Cabin** (reemplazada por HasCabin), **Ticket** (alta cardinalidad y formato heterogéneo), **Name** (identificador de texto libre) y **PassengerId** (identificador sin señal predictiva). Ninguna aporta señal tabular directa al modelo.

In [ ]:
columnas_a_borrar = ['Cabin', 'Ticket', 'Name', 'PassengerId']
datos = datos.drop(columns=columnas_a_borrar)

print('Tamaño despues de borrar:', datos.shape)
print('Columnas que quedaron:', list(datos.columns))

## 3. Codificación (encoding)
- **Sex y Embarked:** one-hot (drop_first=True para evitar multicolinealidad).
- **Pclass:** se trata como ordinal numérica (1 < 2 < 3 tiene orden socioeconómico real) y además se conserva su valor numérico para el escalado. Se documenta la alternativa one-hot si el modelo no admite ordinalidad.

In [ ]:
datos = pd.get_dummies(datos, columns=['Sex', 'Embarked'], drop_first=True)

print('Columnas con dummies:')
print(list(datos.columns))
datos.head(3)

## 4. Escalado
StandardScaler (media 0, desviación 1) sobre las numéricas Age, Fare, SibSp y Parch. Necesario para PCA y modelos basados en distancia. Survived y las dummies binarias no se escalan.

In [ ]:
columnas_escalar = ['Age', 'Fare', 'SibSp', 'Parch']

escalador = StandardScaler()
datos[columnas_escalar] = escalador.fit_transform(datos[columnas_escalar])

print('Estadisticas de columnas escaladas:')
print(datos[columnas_escalar].describe().T.round(3))

## 5. PCA
Se ajusta PCA con 2 componentes para visualización y se reporta la varianza explicada (tabla + gráfico de codo). Scatter PC1 vs PC2 coloreado por Survived y tabla de loadings para interpretar cada componente.

In [ ]:
columnas_x = [col for col in datos.columns if col != 'Survived']
matriz_x = datos[columnas_x].values

modelo_pca = PCA(n_components=2)
componentes = modelo_pca.fit_transform(matriz_x)

varianza_explicada = modelo_pca.explained_variance_ratio_
varianza_acumulada = varianza_explicada.cumsum()

print('Varianza explicada:', np.round(varianza_explicada, 4))
print('Varianza acumulada:', np.round(varianza_acumulada, 4))

tabla_varianza = pd.DataFrame({
    'Componente': ['PC1', 'PC2'],
    'Varianza_Explicada': varianza_explicada,
    'Varianza_Acumulada': varianza_acumulada
})
print(tabla_varianza)

pca_completo = PCA().fit(matriz_x)
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(pca_completo.explained_variance_ratio_) + 1),
         pca_completo.explained_variance_ratio_, marker='o', color='purple')
plt.xlabel('Numero de componente')
plt.ylabel('Varianza explicada')
plt.title('Grafico de codo (PCA)')
plt.tight_layout()
plt.savefig(os.path.join(carpeta_imagenes, 'pca_codo.png'))
plt.show()

plt.figure(figsize=(7, 5))
grafico_dispersion = plt.scatter(componentes[:, 0], componentes[:, 1], c=datos['Survived'].values,
                                 alpha=0.6, cmap='coolwarm')
plt.colorbar(grafico_dispersion, label='Survived')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PC1 vs PC2 por Survived')
plt.tight_layout()
plt.savefig(os.path.join(carpeta_imagenes, 'pca_scatter.png'))
plt.show()

tabla_pesos = pd.DataFrame(modelo_pca.components_.T, index=columnas_x, columns=['PC1', 'PC2']).sort_values('PC1')
print('Pesos de cada variable (loadings):')
print(tabla_pesos.round(3))

plt.figure(figsize=(7, 4))
tabla_pesos['PC1'].plot(kind='barh', color='teal')
plt.title('Pesos en PC1')
plt.tight_layout()
plt.savefig(os.path.join(carpeta_imagenes, 'pca_loadings.png'))
plt.show()

## 6. Exportación
Se guarda el dataset preprocesado en `data/titanic_preprocesado.csv` y se verifica shape y primeras filas.

In [ ]:
datos.to_csv(ruta_salida, index=False)
print('Guardado correctamente en:', ruta_salida)
print('Tamaño final:', datos.shape)
datos.head()